In [2]:
from pathlib import Path

# ===== Input =====
DATA_DIR = Path("/home/danila/networks/data")
AUDIO_DIR = DATA_DIR / "audio"  # mp3
TRAIN_CSV = DATA_DIR / "train_split.csv"
VALID_CSV = DATA_DIR / "valid_split.csv"

# ===== Output: transcripts + segments =====
OUT_TEXT_DIR = DATA_DIR / "text"
OUT_SEG_DIR  = DATA_DIR / "text_whisper_large_v3_segments"
OUT_TEXT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SEG_DIR.mkdir(parents=True, exist_ok=True)

# ===== Output: embeddings =====
OUT_EMB_DIR = DATA_DIR / "embeddings" / "text_gte_base_whisper_large_v3_cls128_v3"
OUT_EMB_DIR.mkdir(parents=True, exist_ok=True)

# ===== Models =====
WHISPER_MODEL_SIZE = "large-v3"
WHISPER_DEVICE = "cuda"
WHISPER_COMPUTE_TYPE = "float16"
WHISPER_BEAM_SIZE = 5
WHISPER_LANGUAGE = "en"
WHISPER_VAD_FILTER = True            # silence removal
MAX_SEC = 20.0

GTE_MODEL_ID = "Alibaba-NLP/gte-base-en-v1.5"
MAX_TOKENS = 128
BATCH_SIZE_EMB = 128

# ===== Misc =====
ID_WIDTH = 5
SAVE_DTYPE = "float32"

In [3]:
import os, json, time, math
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
torch.backends.cuda.matmul.allow_tf32 = True # Amper+ only

In [4]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})

train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

all_ids = sorted(set(train_df["Filename"].tolist()) | set(valid_df["Filename"].tolist()))
print("IDs:", len(all_ids))

def audio_path_for_id(vid: str) -> Path:
    return AUDIO_DIR / f"{vid}.mp3"

missing_audio = [vid for vid in all_ids if not audio_path_for_id(vid).exists()]
print("Missing audio:", len(missing_audio))
if len(missing_audio) and len(missing_audio) < 30:
    print(missing_audio[:30])

IDs: 12660
Missing audio: 0


In [5]:
from faster_whisper import WhisperModel

whisper = WhisperModel(
    WHISPER_MODEL_SIZE,
    device=WHISPER_DEVICE,
    compute_type=WHISPER_COMPUTE_TYPE,
)

print("Whisper ready:", WHISPER_MODEL_SIZE)

Whisper ready: large-v3


In [ ]:
def transcribe_one(mp3_path: Path):
    """
    Returns:
      full_text: str
      segments: list[dict] with start,end,text
      info: dict
    """
    segments_iter, info = whisper.transcribe(
	    str(mp3_path),
	    beam_size=5,
	    language="en",
	    vad_filter=True,
	    condition_on_previous_text=False,
	)

    segments = []
    texts = []
    for s in segments_iter:
        if s.start >= MAX_SEC:
            break
        if s.end <= 0:
            continue
        txt = s.text.strip()
        if txt:
            texts.append(txt)

    full_text = " ".join(texts).strip()

    info_dict = {
        "language": getattr(info, "language", None),
        "language_probability": float(getattr(info, "language_probability", 0.0) or 0.0),
        "duration": float(getattr(info, "duration", 0.0) or 0.0),
    }
    return full_text, segments, info_dict

failed = []
metas = []

for vid in tqdm(all_ids, desc="Transcribe Whisper large-v3"):
    mp3 = audio_path_for_id(vid)
    if not mp3.exists():
        failed.append({"video_id": vid, "stage": "audio_missing"})
        tqdm.write(f"[MISS] {vid}: audio missing")
        continue

    out_txt = OUT_TEXT_DIR / f"{vid}.txt"
    out_seg = OUT_SEG_DIR / f"{vid}.json"

    # resume
    if out_txt.exists() and out_seg.exists():
        metas.append({"video_id": vid, "status": "skipped_existing"})
        continue

    try:
        t0 = time.time()
        full_text, segments, info = transcribe_one(mp3)
        dt = time.time() - t0

        out_txt.write_text(full_text, encoding="utf-8")
        out_seg.write_text(json.dumps({"video_id": vid, "info": info, "segments": segments}, ensure_ascii=False), encoding="utf-8")

        metas.append({
            "video_id": vid,
            "status": "ok",
            "chars": len(full_text),
            "num_segments": len(segments),
            "duration": info.get("duration", 0.0),
            "lang": info.get("language", None),
            "lang_p": info.get("language_probability", 0.0),
            "sec": dt,
        })
    except Exception as e:
        failed.append({"video_id": vid, "stage": "transcribe", "error": repr(e)})
        tqdm.write(f"[ERR] {vid}: {repr(e)}")

meta_df = pd.DataFrame(metas)
fail_df = pd.DataFrame(failed)

RUN_DIR = DATA_DIR / "runs" / "whisper_large_v3_transcribe_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)

meta_df.to_parquet(RUN_DIR / "transcribe_meta.parquet", index=False)
if len(fail_df):
    fail_df.to_csv(RUN_DIR / "transcribe_failed.csv", index=False)

tqdm.write(f"DONE transcribe | ok={(meta_df['status']=='ok').sum()} skipped={(meta_df['status']=='skipped_existing').sum()} failed={len(fail_df)}")
tqdm.write(f"Meta: {RUN_DIR / 'transcribe_meta.parquet'}")
meta_df.to_parquet(RUN_DIR / "transcribe_meta.parquet", index=False)
if len(fail_df):
    fail_df.to_csv(RUN_DIR / "transcribe_failed.csv", index=False)

tqdm.write(f"DONE transcribe | ok={(meta_df['status']=='ok').sum()} skipped={(meta_df['status']=='skipped_existing').sum()} failed={len(fail_df)}")
tqdm.write(f"Meta: {RUN_DIR / 'transcribe_meta.parquet'}")

Transcribe Whisper large-v3:   1%|█                                                                                                             | 129/12660 [01:21<2:18:07,  1.51it/s]

In [6]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained(GTE_MODEL_ID)
model = AutoModel.from_pretrained(GTE_MODEL_ID, trust_remote_code=True).to("cuda").eval()
model.eval()

print("GTE ready:", GTE_MODEL_ID)

@torch.inference_mode()
def encode_cls_norm(texts, max_tokens=128):
    batch = tokenizer(texts, padding=True, truncation=True, max_length=max_tokens, return_tensors="pt").to("cuda")
    out = model(**batch)
    emb = out.last_hidden_state[:, 0, :]
    emb = F.normalize(emb, p=2, dim=1)
    return emb.float()

GTE ready: Alibaba-NLP/gte-base-en-v1.5


In [7]:
def text_path_for_id(vid: str) -> Path:
    return OUT_TEXT_DIR / f"{vid}.txt"

def emb_path_for_id(vid: str) -> Path:
    return OUT_EMB_DIR / f"{vid}.npz"

def count_tokens(text: str, max_tokens: int):
    ids = tokenizer(text, truncation=True, max_length=max_tokens, add_special_tokens=True)["input_ids"]
    return len(ids)

todo = []
for vid in all_ids:
    txtp = text_path_for_id(vid)
    embp = emb_path_for_id(vid)
    if not txtp.exists():
        continue
    if embp.exists():
        continue
    todo.append(vid)

print("To embed:", len(todo))

failed_emb = []
meta_emb = []

batch_ids = []
batch_texts = []

for vid in tqdm(todo, desc="GTE embeddings"):
    try:
        text = text_path_for_id(vid).read_text(encoding="utf-8").strip()
    except Exception as e:
        failed_emb.append({"video_id": vid, "stage": "read_txt", "error": repr(e)})
        tqdm.write(f"[ERR] read {vid}: {repr(e)}")
        continue

    batch_ids.append(vid)
    batch_texts.append(text)

    if len(batch_ids) >= BATCH_SIZE_EMB:
        try:
            emb = encode_cls_norm(batch_texts, max_tokens=MAX_TOKENS).detach().cpu().numpy()
            for i, v in enumerate(batch_ids):
                out = emb_path_for_id(v)
                dtype = np.float32 if SAVE_DTYPE == "float32" else np.float16
                vec = emb[i].astype(dtype)

                num_tok = count_tokens(batch_texts[i], MAX_TOKENS)
                np.savez(
                    out,
                    embedding=vec,
                    model_id=np.array([GTE_MODEL_ID]),
                    max_tokens=np.array([MAX_TOKENS], dtype=np.int32),
                    num_chars=np.array([len(batch_texts[i])], dtype=np.int32),
                    num_tokens_used=np.array([num_tok], dtype=np.int32),
                    pooling=np.array(["cls"]),
                )
                meta_emb.append({"video_id": v, "status": "ok", "dim": int(vec.shape[0]), "num_tokens_used": num_tok})
        except Exception as e:
            for v in batch_ids:
                failed_emb.append({"video_id": v, "stage": "embed_batch", "error": repr(e)})
            tqdm.write(f"[ERR] embed batch: {repr(e)}")

        batch_ids, batch_texts = [], []

# flush tail
if batch_ids:
    try:
        emb = encode_cls_norm(batch_texts, max_tokens=MAX_TOKENS).detach().cpu().numpy()
        for i, v in enumerate(batch_ids):
            out = emb_path_for_id(v)
            dtype = np.float32 if SAVE_DTYPE == "float32" else np.float16
            vec = emb[i].astype(dtype)

            num_tok = count_tokens(batch_texts[i], MAX_TOKENS)
            np.savez(
                out,
                embedding=vec,
                model_id=np.array([GTE_MODEL_ID]),
                max_tokens=np.array([MAX_TOKENS], dtype=np.int32),
                num_chars=np.array([len(batch_texts[i])], dtype=np.int32),
                num_tokens_used=np.array([num_tok], dtype=np.int32),
                pooling=np.array(["cls"]),
            )
            meta_emb.append({"video_id": v, "status": "ok", "dim": int(vec.shape[0]), "num_tokens_used": num_tok})
    except Exception as e:
        for v in batch_ids:
            failed_emb.append({"video_id": v, "stage": "embed_batch", "error": repr(e)})
        tqdm.write(f"[ERR] embed tail: {repr(e)}")

meta_emb_df = pd.DataFrame(meta_emb)
fail_emb_df = pd.DataFrame(failed_emb)

meta_emb_df.to_parquet(RUN_DIR / "emb_meta.parquet", index=False)
if len(fail_emb_df):
    fail_emb_df.to_csv(RUN_DIR / "emb_failed.csv", index=False)

tqdm.write(f"DONE embed | ok={len(meta_emb_df)} failed={len(fail_emb_df)}")
tqdm.write(f"Emb dir: {OUT_EMB_DIR}")

To embed: 12660


GTE embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12660/12660 [00:31<00:00, 400.36it/s]


NameError: name 'RUN_DIR' is not defined

In [24]:
for vid in all_ids[:20]:
    p = emb_path_for_id(vid)
    if p.exists():
        d = np.load(p, allow_pickle=False)
        print("Example:", vid, "dim:", d["embedding"].shape, "tokens:", int(d["num_tokens_used"][0]), "dtype:", d["embedding"].dtype)
        break

Example: 00000 dim: (768,) tokens: 20 dtype: float32


In [25]:
rows = []
for p in OUT_EMB_DIR.glob("*.npz"):
    vid = p.stem
    d = np.load(p, allow_pickle=False)
    rows.append({
        "video_id": vid,
        "dim": int(d["embedding"].shape[0]),
        "num_tokens_used": int(d["num_tokens_used"][0]),
        "max_tokens": int(d["max_tokens"][0]),
        "pooling": str(d["pooling"][0]),
        "model_id": str(d["model_id"][0]),
    })

index_df = pd.DataFrame(rows).sort_values("video_id").reset_index(drop=True)
index_path = RUN_DIR / "emb_index.parquet"
index_df.to_parquet(index_path, index=False)
print("Saved:", index_path, "| rows:", len(index_df))
index_df.head()

Saved: /home/danila/networks/data/runs/whisper_large_v3_transcribe_v1/emb_index.parquet | rows: 12660


,video_id,dim,num_tokens_used,max_tokens,pooling,model_id
0,00000,768,20,128,cls,Alibaba-NLP/gte-base-en-v1.5
1,00001,768,17,128,cls,Alibaba-NLP/gte-base-en-v1.5
2,00002,768,36,128,cls,Alibaba-NLP/gte-base-en-v1.5
3,00003,768,25,128,cls,Alibaba-NLP/gte-base-en-v1.5
4,00004,768,16,128,cls,Alibaba-NLP/gte-base-en-v1.5
